# PharmaCon — Real Handwritten Medical OCR

## Clean training track

This notebook intentionally contains ONLY the real-handwriting experiment.

Pipeline:

1. Install dependencies
2. Load Florence-2 Large
3. Locate/extract the original Doctor's Handwritten Prescription BD dataset
4. Inspect real handwritten samples
5. Build PyTorch datasets and DataLoaders
6. Run a small baseline inference test
7. Train a medical-handwriting LoRA adapter
8. Save the adapter
9. Evaluate on the held-out test split
10. Compare baseline vs adapted OCR

**Important:** the public dataset contains cropped handwritten medicine-name words, not complete prescription pages. It is therefore suitable for establishing medical-handwriting OCR capability, not yet for claiming full prescription-field extraction.


## 1. Install dependencies

In [ ]:
!pip install -q "transformers>=4.46.0" einops timm peft pillow opencv-python numpy scipy scikit-learn matplotlib levenshtein datasets jiwer
!pip install -q -U "torchao>=0.16.0"


## 2. Imports and GPU check

In [ ]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from PIL import Image
from torch.utils.data import Dataset, DataLoader

from transformers import AutoProcessor, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Florence-2 setup

This cell creates the model from scratch. It does **not** depend on `ocr_engine` or any other object from the previous notebook.


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.configuration_utils import PretrainedConfig
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
from transformers.dynamic_module_utils import get_imports
from unittest.mock import patch

MODEL_ID = "microsoft/florence-2-large"

# Patches for modern transformers compatibility
PretrainedConfig.forced_bos_token_id = None
PreTrainedTokenizerBase.additional_special_tokens = property(
    lambda self: self.special_tokens_map.get("additional_special_tokens", [])
)
torch.nn.Module._supports_sdpa = True

print(f"Loading {MODEL_ID}...")

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

def fixed_get_imports(filename):
    imports = get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports

with patch("transformers.dynamic_module_utils.get_imports", fixed_get_imports):
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        attn_implementation="eager",
        trust_remote_code=True
    ).to(device)

base_model.eval()

print("✓ Florence-2 loaded")


## 4. Dataset setup

The notebook expects the original Kaggle ZIP or an already extracted dataset.

Expected extracted structure:

```text
Doctor’s Handwritten Prescription BD dataset/
├── Training/
│   ├── training_labels.csv
│   └── training_words/
├── Validation/
│   ├── validation_labels.csv
│   └── validation_words/
└── Testing/
    ├── testing_labels.csv
    └── testing_words/
```


In [ ]:
import zipfile

DATA_ROOT = Path(
    "/content/doctor_handwriting/Doctor’s Handwritten Prescription BD dataset"
)

if not DATA_ROOT.exists():
    print("Dataset not found at:")
    print(DATA_ROOT)
    print()
    print("Upload the original Kaggle ZIP now.")

    from google.colab import files

    uploaded = files.upload()

    zip_candidates = [
        Path("/content") / name
        for name in uploaded
        if name.lower().endswith(".zip")
    ]

    if not zip_candidates:
        raise FileNotFoundError(
            "No ZIP file was uploaded."
        )

    zip_path = zip_candidates[0]
    extract_root = Path("/content/doctor_handwriting")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_root)

    # Find the dataset root dynamically.
    matches = list(
        extract_root.rglob("training_labels.csv")
    )

    if not matches:
        raise FileNotFoundError(
            "training_labels.csv was not found after extraction."
        )

    DATA_ROOT = matches[0].parent.parent

print("✓ Dataset root:")
print(DATA_ROOT)

required_paths = [
    DATA_ROOT / "Training" / "training_labels.csv",
    DATA_ROOT / "Training" / "training_words",
    DATA_ROOT / "Validation" / "validation_labels.csv",
    DATA_ROOT / "Validation" / "validation_words",
    DATA_ROOT / "Testing" / "testing_labels.csv",
    DATA_ROOT / "Testing" / "testing_words",
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing expected dataset path: {path}"
        )

print("✓ Dataset structure verified")


## 5. Inspect annotations

In [ ]:
for split in ["Training", "Validation", "Testing"]:
    csv_path = DATA_ROOT / split / {
        "Training": "training_labels.csv",
        "Validation": "validation_labels.csv",
        "Testing": "testing_labels.csv"
    }[split]

    df = pd.read_csv(csv_path)

    print(f"\n=== {split} ===")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(10))

train_df = pd.read_csv(
    DATA_ROOT / "Training" / "training_labels.csv"
)

print(
    "\nUnique medicine names:",
    train_df["MEDICINE_NAME"].nunique()
)

print(
    "Unique generic names:",
    train_df["GENERIC_NAME"].nunique()
)


## 6. Inspect real handwriting

Do not continue to training until you visually confirm that these are genuine handwritten medicine-name crops.


In [ ]:
train_image_dir = DATA_ROOT / "Training" / "training_words"

sample_indices = [
    0, 10, 50, 100,
    200, 500, 1000, 2000
]

fig, axes = plt.subplots(
    2, 4,
    figsize=(16, 7)
)

for ax, idx in zip(axes.flat, sample_indices):

    row = train_df.iloc[idx]

    image_path = train_image_dir / str(row["IMAGE"])
    image = Image.open(image_path).convert("RGB")

    ax.imshow(image)
    ax.set_title(
        f"Medicine: {row['MEDICINE_NAME']}\n"
        f"Generic: {row['GENERIC_NAME']}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()


## 7. PyTorch dataset

The ground-truth transcription is `MEDICINE_NAME`, exactly as provided by the dataset.


In [ ]:
class HandwritingOCRDataset(Dataset):

    def __init__(
        self,
        image_dir,
        csv_path,
        processor,
        max_length=64
    ):
        self.image_dir = Path(image_dir)
        self.processor = processor
        self.max_length = max_length

        self.data = pd.read_csv(csv_path)

        self.data.columns = [
            c.strip().upper()
            for c in self.data.columns
        ]

        required = {
            "IMAGE",
            "MEDICINE_NAME",
            "GENERIC_NAME"
        }

        missing = required - set(self.data.columns)

        if missing:
            raise ValueError(
                f"Missing columns: {missing}"
            )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        image_path = (
            self.image_dir /
            str(row["IMAGE"])
        )

        if not image_path.exists():
            raise FileNotFoundError(
                f"Image not found: {image_path}"
            )

        image = Image.open(
            image_path
        ).convert("RGB").resize((768, 768))

        text = str(
            row["MEDICINE_NAME"]
        ).strip()

        inputs = self.processor(
            text="<OCR>",
            images=image,
            return_tensors="pt"
        )

        input_ids = (
            inputs["input_ids"]
            .squeeze(0)
        )

        pixel_values = (
            inputs["pixel_values"]
            .squeeze(0)
        )

        labels = self.processor.tokenizer(
            text=text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze(0)

        pad_token_id = (
            self.processor.tokenizer.pad_token_id
        )

        if pad_token_id is not None:
            labels[
                labels == pad_token_id
            ] = -100

        return {
            "input_ids": input_ids,
            "pixel_values": pixel_values,
            "labels": labels,
            "text": text,
            "image_path": str(image_path)
        }


train_dataset = HandwritingOCRDataset(
    DATA_ROOT / "Training" / "training_words",
    DATA_ROOT / "Training" / "training_labels.csv",
    processor
)

validation_dataset = HandwritingOCRDataset(
    DATA_ROOT / "Validation" / "validation_words",
    DATA_ROOT / "Validation" / "validation_labels.csv",
    processor
)

test_dataset = HandwritingOCRDataset(
    DATA_ROOT / "Testing" / "testing_words",
    DATA_ROOT / "Testing" / "testing_labels.csv",
    processor
)

print("✓ Datasets created")
print("Training:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Testing:", len(test_dataset))

sample = train_dataset[0]

print("\nSample check")
print("Ground truth:", sample["text"])
print("Image path:", sample["image_path"])
print("Pixel values:", sample["pixel_values"].shape)
print("Input IDs:", sample["input_ids"].shape)
print("Labels:", sample["labels"].shape)


## 8. DataLoaders

In [ ]:
def collate_fn(batch):
    return {
        "input_ids": torch.stack([
            x["input_ids"] for x in batch
        ]),
        "pixel_values": torch.stack([
            x["pixel_values"] for x in batch
        ]),
        "labels": torch.stack([
            x["labels"] for x in batch
        ]),
        "text": [
            x["text"] for x in batch
        ],
        "image_path": [
            x["image_path"] for x in batch
        ]
    }


BATCH_SIZE = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("✓ DataLoaders ready")
print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Testing batches:", len(test_loader))


## 9. Baseline inference on real handwriting

Before training, we test the untouched Florence-2 model on a few real handwritten images.
This is NOT the final benchmark; it is a smoke test to confirm the model actually produces text.


In [ ]:
from jiwer import cer, wer

# =========================================================
# 9. BASELINE EVALUATION — BEFORE LoRA
# =========================================================

def clean_ocr_text(text):
    return " ".join(text.split()).strip()


def predict_image(model, image, max_new_tokens=64):
    model.eval()
    image_square = image.resize((768, 768))
    prompt = "<OCR>"
    inputs = processor(text=prompt, images=image_square, return_tensors="pt")
    input_ids = inputs["input_ids"].to(device)
    pixel_values = inputs["pixel_values"].to(device=device, dtype=base_model.dtype)

    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=input_ids,
            pixel_values=pixel_values,
            max_new_tokens=max_new_tokens,
            num_beams=3,
            do_sample=False,
            use_cache=False
        )

    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed_answer = processor.post_process_generation(generated_text, task=prompt, image_size=(768, 768))
    return clean_ocr_text(parsed_answer.get(prompt, ""))


print("=" * 70)
print("BASELINE FLORENCE-2 — REAL HANDWRITTEN TEST SET")
print("=" * 70)

baseline_predictions = []
baseline_ground_truths = []

for idx in range(len(test_dataset)):

    row = test_dataset.data.iloc[idx]

    image_path = (
        test_dataset.image_dir /
        str(row["IMAGE"])
    )

    image = Image.open(
        image_path
    ).convert("RGB")

    prediction = predict_image(
        base_model,
        image
    )

    truth = str(
        row["MEDICINE_NAME"]
    ).strip()

    baseline_predictions.append(prediction)
    baseline_ground_truths.append(truth)

    # Only print first 20 examples
    if idx < 20:

        print(
            f"\nExample {idx + 1}"
        )

        print(
            "GT:  ",
            truth
        )

        print(
            "Pred:",
            prediction
        )


# ---------------------------------------------------------
# Baseline metrics
# ---------------------------------------------------------

baseline_exact = np.mean([
    p.lower().strip()
    ==
    g.lower().strip()

    for p, g in zip(
        baseline_predictions,
        baseline_ground_truths
    )
])

baseline_cer = np.mean([
    cer(g, p)

    for p, g in zip(
        baseline_predictions,
        baseline_ground_truths
    )
])

baseline_wer = np.mean([
    wer(g, p)

    for p, g in zip(
        baseline_predictions,
        baseline_ground_truths
    )
])


print("\n" + "=" * 70)
print("BASELINE RESULTS")
print("=" * 70)

print(
    f"Exact Match: {baseline_exact:.2%}"
)

print(
    f"CER:         {baseline_cer:.2%}"
)

print(
    f"WER:         {baseline_wer:.2%}"
)

print("=" * 70)

## 10. Create the medical-handwriting LoRA

This is a general adapter for **real medical handwriting**.

It is separate from the future doctor-specific writer adapter.


In [ ]:
# Keep the base model untouched.
# get_peft_model creates a new LoRA-wrapped model.

medical_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

medical_model = get_peft_model(
    base_model,
    medical_lora_config
)

medical_model.print_trainable_parameters()

print("✓ Medical-handwriting LoRA created")


## 11. Train the medical-handwriting LoRA

Start with 1 epoch as a smoke test. Once that completes with finite loss, you can change `EPOCHS` to 3.

The code intentionally stops on NaN/Inf loss.


In [ ]:
EPOCHS = 3
LEARNING_RATE = 5e-5

def train_one_epoch(model, loader, optimizer, scaler, use_amp):

    model.train()

    total_loss = 0.0
    steps = 0

    for batch_idx, batch in enumerate(loader):

        input_ids = batch["input_ids"].to(device)

        pixel_values = batch["pixel_values"].to(
            device=device,
            dtype=torch.float16 if use_amp
            else torch.float32
        )

        labels = batch["labels"].to(device)

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.cuda.amp.autocast(
            dtype=torch.float16,
            enabled=use_amp
        ):

            outputs = model(
                input_ids=input_ids,
                pixel_values=pixel_values,
                labels=labels
            )

            loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at batch "
                f"{batch_idx + 1}: {loss.item()}"
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            [
                p for p in model.parameters()
                if p.requires_grad
            ],
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        total_loss += float(
            loss.detach().cpu()
        )

        steps += 1

        if (batch_idx + 1) % 100 == 0:
            print(
                f"  Batch {batch_idx + 1}/{len(loader)} "
                f"- Loss: {loss.item():.4f}"
            )

    return total_loss / max(steps, 1)


trainable_params = [
    p for p in medical_model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=0.01
)

use_amp = torch.cuda.is_available()

scaler = torch.cuda.amp.GradScaler(
    enabled=use_amp
)

training_history = []

for epoch in range(EPOCHS):

    avg_loss = train_one_epoch(
        medical_model,
        train_loader,
        optimizer,
        scaler,
        use_amp
    )

    training_history.append(
        float(avg_loss)
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS} "
        f"- Train Loss: {avg_loss:.4f}"
    )

medical_model.eval()

print("✓ Training completed")


## 12. Save the medical-handwriting adapter

In [ ]:
MEDICAL_ADAPTER_DIR = Path(
    "/content/pharmacon_medical_handwriting_adapter"
)

MEDICAL_ADAPTER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

medical_model.save_pretrained(
    MEDICAL_ADAPTER_DIR
)

with open(
    MEDICAL_ADAPTER_DIR / "metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "base_model": MODEL_ID,
            "dataset": "Doctor's Handwritten Prescription BD",
            "train_samples": len(train_dataset),
            "validation_samples": len(validation_dataset),
            "test_samples": len(test_dataset),
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "training_loss": training_history
        },
        f,
        indent=2
    )

print(
    f"✓ Adapter saved to {MEDICAL_ADAPTER_DIR}"
)


## 13. Evaluate the adapted model on the full held-out test split

In [ ]:
from jiwer import cer, wer




predictions = []
ground_truths = []

for idx in range(len(test_dataset)):

    row = test_dataset.data.iloc[idx]

    image_path = (
        test_dataset.image_dir /
        str(row["IMAGE"])
    )

    image = Image.open(
        image_path
    ).convert("RGB")

    prediction = predict_image(
        medical_model,
        image
    )

    truth = str(
        row["MEDICINE_NAME"]
    ).strip()

    predictions.append(prediction)
    ground_truths.append(truth)

    if idx < 20:
        print(
            f"\nExample {idx + 1}"
            f"\nGT:   {truth}"
            f"\nPred: {prediction}"
        )


exact_match = np.mean([
    p.lower().strip()
    ==
    g.lower().strip()
    for p, g in zip(
        predictions,
        ground_truths
    )
])

mean_cer = np.mean([
    cer(g, p)
    for p, g in zip(
        predictions,
        ground_truths
    )
])

mean_wer = np.mean([
    wer(g, p)
    for p, g in zip(
        predictions,
        ground_truths
    )
])

print("\n" + "=" * 70)
print("REAL HANDWRITTEN MEDICAL OCR RESULTS")
print("=" * 70)
print("Test samples:", len(ground_truths))
print(f"Exact Match: {exact_match:.2%}")
print(f"CER:         {mean_cer:.2%}")
print(f"WER:         {mean_wer:.2%}")
print("=" * 70)


## 14. Compare baseline and adapted model on the same test samples

This is a true before/after comparison. The baseline and adapted model see the same held-out images.


In [ ]:
## 14. BASELINE vs MEDICAL-HANDWRITING ADAPTER

print("=" * 70)
print("BASELINE vs MEDICAL-HANDWRITING ADAPTER")
print("=" * 70)

# baseline_predictions were collected BEFORE LoRA training.
# Now use the trained medical_model on the exact same test images.

adapted_predictions = []

for idx in range(len(test_dataset)):

    row = test_dataset.data.iloc[idx]

    image_path = (
        test_dataset.image_dir /
        str(row["IMAGE"])
    )

    image = Image.open(
        image_path
    ).convert("RGB")

    prediction = predict_image(
        medical_model,
        image
    )

    adapted_predictions.append(
        prediction
    )


# ---------------------------------------------------------
# Adapted metrics
# ---------------------------------------------------------

adapted_exact = np.mean([
    p.lower().strip() == g.lower().strip()
    for p, g in zip(
        adapted_predictions,
        baseline_ground_truths
    )
])

adapted_cer = np.mean([
    cer(g, p)
    for p, g in zip(
        adapted_predictions,
        baseline_ground_truths
    )
])

adapted_wer = np.mean([
    wer(g, p)
    for p, g in zip(
        adapted_predictions,
        baseline_ground_truths
    )
])


# ---------------------------------------------------------
# Comparison
# ---------------------------------------------------------

print(
    f"\nBaseline Exact Match:   "
    f"{baseline_exact:.2%}"
)

print(
    f"Adapted Exact Match:    "
    f"{adapted_exact:.2%}"
)

print(
    f"Baseline CER:            "
    f"{baseline_cer:.2%}"
)

print(
    f"Adapted CER:             "
    f"{adapted_cer:.2%}"
)

print(
    f"Baseline WER:            "
    f"{baseline_wer:.2%}"
)

print(
    f"Adapted WER:             "
    f"{adapted_wer:.2%}"
)

print(
    f"\nExact Match Gain:        "
    f"{(adapted_exact - baseline_exact) * 100:+.2f} pp"
)

print(
    f"CER Change:              "
    f"{(adapted_cer - baseline_cer) * 100:+.2f} pp"
)

print(
    f"WER Change:              "
    f"{(adapted_wer - baseline_wer) * 100:+.2f} pp"
)

print("=" * 70)

## 15. What this result means

If the adapted model improves over the untouched Florence-2 baseline, we have demonstrated that
real handwritten medical OCR can be learned with parameter-efficient adaptation.

This still does **not** mean the system can read a complete prescription page.

The next research phase is:

```text
Medical-handwriting model
        ↓
Doctor-specific calibration samples
        ↓
Writer-specific LoRA
        ↓
Unseen handwriting from that doctor
```

That is where PharmaCon's writer-personalization contribution is evaluated.


### Reference dataset information

The upstream Doctor's Handwritten Prescription BD dataset contains 4,680 segmented handwritten prescription words and 78 medicine classes, with published train/test/validation splits of 3,120/780/780. The Hugging Face Lance conversion documents the same 4,680 images and split sizes. The current notebook intentionally uses the original extracted Kaggle files instead of the broken Lance loader.
